# Querying UniProt programmatically with Python

[UniProt](https://www.uniprot.org/) is the central public resource for protein
sequences and functional annotation. Everything you can do on the website you can
also do over its **REST API** — and that's what lets you build reproducible,
scriptable pipelines instead of clicking and downloading by hand.

This tutorial teaches the API using **only the [`requests`](https://requests.readthedocs.io/)
library** — no high-level wrappers (`unipressed`, `bioservices`, …). Wrappers are
convenient, but understanding the raw HTTP calls means you can debug anything, read
the official docs directly, and port the ideas to any language.

**Prerequisites:** intermediate Python and a basic feel for `requests`
(`requests.get(...)`, `.status_code`, `.text`, `.json()`).

**What we'll cover, building from one-liners up to reusable functions:**

1. The anatomy of a UniProt request — endpoint and parameters
2. Query syntax (fields, ranges, booleans)
3. Output formats — JSON, TSV, FASTA, list
4. Selecting only the fields you need
5. **Pagination** with the `Link` header (cursor-based)
6. **Robust** requests — a `Session`, a `User-Agent`, retries/backoff
7. The **stream** endpoint for bulk downloads
8. The asynchronous **ID-mapping** API
9. Handy snippets (single-entry fetch, count-only queries, other datasets)
10. **Capstone:** pull FASTA-header fields as JSON, view them, and write a `.fasta`


## 0. Set up the environment (one-time)

In [1]:
# Run ONCE, then select the "Python (uniprot-tutorial)" kernel

# !mamba env create -f envs/uniprot-tutorial.yml
# !python -m ipykernel install --user --name uniprot-tutorial --display-name "uniprot-tutorial"


## 1. Imports

Everything we need is the standard library plus `requests` and `pandas`. We also pin
the API base URL in one place.

In [2]:
import io      # wrap text responses so pandas can read them
import re      # parse the pagination "Link" header
import time    # backoff / polling
import json    # pretty-printing

import requests
import pandas as pd

API = "https://rest.uniprot.org"

print("requests", requests.__version__, "| pandas", pd.__version__)

requests 2.34.2 | pandas 2.3.3


## 2. Anatomy of a request

Almost every search hits one endpoint:

```
https://rest.uniprot.org/uniprotkb/search
```

and is shaped by a handful of query parameters:

| parameter | meaning | example |
|-----------|---------|---------|
| `query`   | the search expression | `insulin AND organism_id:9606` |
| `format`  | response format | `json`, `tsv`, `fasta`, `list`, `xml` |
| `fields`  | which columns/fields to return | `accession,gene_names,length` |
| `size`    | results per page (default 25, **max 500**) | `500` |
| `includeIsoform` | include isoform sequences? (**UniProtKB only**) | `true`, `false` |

Two **response headers** matter for scripting:

* `X-Total-Results` — how many hits the query has in total
* `Link` — a URL to the *next* page (cursor-based pagination), present only when
  more pages exist

We pass parameters as a `params=` dict and let `requests` handle URL-encoding.

## 3. The smallest possible request

Search for "insulin", ask for one JSON result, and inspect the response.

In [3]:
r = requests.get(
    f"{API}/uniprotkb/search",
    params={"query": "insulin", "format": "json", "size": 1},
)

print("status code :", r.status_code)
print("request URL :", r.url)               # see exactly what requests built
print("total hits  :", r.headers["x-total-results"])  # header lookups are case-insensitive

first = r.json()["results"][0]
print("first hit   :", first["primaryAccession"], "-", first["uniProtkbId"])

status code : 200
request URL : https://rest.uniprot.org/uniprotkb/search?query=insulin&format=json&size=1
total hits  : 299403
first hit   : P01308 - INS_HUMAN


## 4. Query syntax

The `query` string is a Lucene-style expression. The main building blocks:

* **Free text:** `insulin`
* **Field queries:** `gene:INS`, `organism_id:9606`, `accession:P01308`, `reviewed:true`
* **Ranges:** `length:[100 TO 200]`, `date_created:[2020-01-01 TO *]`
* **Booleans & grouping:** `AND`, `OR`, `NOT`, parentheses

A few high-value fields: `reviewed:true` (Swiss-Prot only), `organism_id:` (NCBI taxon
id — unambiguous, prefer it over names), `gene:`, `keyword:`, `proteome:`, `xref:`.
The full list of query fields is in the
[query-fields docs](https://www.uniprot.org/help/query-fields).

The cell below uses a neat trick — `size=0` returns an **empty body but a full
`X-Total-Results` header** — to count hits for several queries cheaply.

In [4]:
queries = {
    "free text"            : "insulin",
    "gene name"            : "gene:INS",
    "organism by taxon id" : "organism_id:9606",
    "reviewed only"        : "gene:INS AND reviewed:true",
    "by accession"         : "accession:P01308",
    "length range"         : "length:[100 TO 200] AND gene:INS",
    "boolean + grouping"   : "(gene:INS OR gene:INSR) AND organism_id:9606",
}

for label, q in queries.items():
    r = requests.get(f"{API}/uniprotkb/search",
                     params={"query": q, "format": "list", "size": 0})
    print(f"{label:22s} | {q:48s} -> {r.headers['x-total-results']:>7s} hits")

free text              | insulin                                          ->  299403 hits
gene name              | gene:INS                                         ->     759 hits
organism by taxon id   | organism_id:9606                                 ->  205155 hits
reviewed only          | gene:INS AND reviewed:true                       ->      97 hits
by accession           | accession:P01308                                 ->       1 hits
length range           | length:[100 TO 200] AND gene:INS                 ->     575 hits
boolean + grouping     | (gene:INS OR gene:INSR) AND organism_id:9606     ->      10 hits


## 5. Output formats

The same query can come back in different shapes — pick the one that fits the job:

* **`json`** — full nested structure; parse with `r.json()`
* **`tsv`** — flat table; load straight into pandas
* **`fasta`** — sequences with UniProt headers; raw text
* **`list`** — just the accessions, one per line
* also available: `xml`, `gff`, `txt`, `rdf`, `xlsx`

In [5]:
query = "gene:INS AND organism_id:9606 AND reviewed:true"

# JSON -> nested dict
js = requests.get(f"{API}/uniprotkb/search",
                  params={"query": query, "format": "json", "size": 3}).json()
print("JSON  :", [h["primaryAccession"] for h in js["results"]])

# list -> bare accessions
lst = requests.get(f"{API}/uniprotkb/search",
                   params={"query": query, "format": "list", "size": 3}).text
print("list  :", lst.split())

# FASTA -> raw text
fa = requests.get(f"{API}/uniprotkb/search",
                  params={"query": query, "format": "fasta", "size": 1}).text
print("FASTA :")
print(fa)

JSON  : ['P01308', 'F8WCM5']
list  : ['P01308', 'F8WCM5']
FASTA :
>sp|P01308|INS_HUMAN Insulin OS=Homo sapiens OX=9606 GN=INS PE=1 SV=1
MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAED
LQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN



In [6]:
# TSV -> pandas DataFrame. Wrap the text in io.StringIO and read it as tab-separated.
tsv = requests.get(
    f"{API}/uniprotkb/search",
    params={"query": query, "format": "tsv",
            "fields": "accession,id,gene_names,length,organism_name", "size": 5},
).text

df = pd.read_csv(io.StringIO(tsv), sep="\t")
df

,Entry,Entry Name,Gene Names,Length,Organism
0,P01308,INS_HUMAN,INS,110,Homo sapiens (Human)
1,F8WCM5,INSR2_HUMAN,INS-IGF2,200,Homo sapiens (Human)


## 6. Returning only the fields you need

By default UniProt returns large records. The `fields` parameter trims the response to
exactly the columns you want — smaller, faster, and (for TSV) in the order you list.

* For **TSV**, `fields` maps directly to columns.
* For **JSON**, `fields` controls which sections are populated, but the structure stays
  nested (e.g. gene names live under `genes[...].geneName.value`).

The catalogue of return fields is in the
[return-fields docs](https://www.uniprot.org/help/return_fields).

In [7]:
fields = "accession,id,protein_name,gene_names,organism_name,length,mass"

tsv = requests.get(
    f"{API}/uniprotkb/search",
    params={"query": "gene:INS AND reviewed:true",
            "format": "tsv", "fields": fields, "size": 10},
).text

pd.read_csv(io.StringIO(tsv), sep="\t")

,Entry,Entry Name,Protein names,Gene Names,Organism,Length,Mass
0,P01308,INS_HUMAN,Insulin [Cleaved into: Insulin B chain; Insuli...,INS,Homo sapiens (Human),110,11981
1,P01317,INS_BOVIN,Insulin [Cleaved into: Insulin B chain; Insuli...,INS,Bos taurus (Bovine),105,11393
2,P01326,INS2_MOUSE,Insulin-2 [Cleaved into: Insulin-2 B chain; In...,Ins2 Ins-2,Mus musculus (Mouse),110,12364
3,P01322,INS1_RAT,Insulin-1 [Cleaved into: Insulin-1 B chain; In...,Ins1 Ins-1,Rattus norvegicus (Rat),110,12421
4,P01325,INS1_MOUSE,Insulin-1 [Cleaved into: Insulin-1 B chain; In...,Ins1 Ins-1,Mus musculus (Mouse),108,12160
5,P01323,INS2_RAT,Insulin-2 [Cleaved into: Insulin-2 B chain; In...,Ins2 Ins-2,Rattus norvegicus (Rat),110,12339
6,P01315,INS_PIG,Insulin [Cleaved into: Insulin B chain; Insuli...,INS,Sus scrofa (Pig),108,11672
7,P67970,INS_CHICK,Insulin [Cleaved into: Insulin B chain; Insuli...,INS,Gallus gallus (Chicken),107,11981
8,Q91XI3,INS_ICTTR,Insulin [Cleaved into: Insulin B chain; Insuli...,INS,Ictidomys tridecemlineatus (Thirteen-lined gro...,110,12004
9,P01321,INS_CANLF,Insulin [Cleaved into: Insulin B chain; Insuli...,INS,Canis lupus familiaris (Dog) (Canis familiaris),110,12190


## 7. Pagination — following the `Link` header

A search returns at most `size` (≤ 500) results per page. To get everything, UniProt
uses **cursor-based pagination**: each response carries a `Link` header pointing at the
next page. You keep following it until it disappears.

```
Link: <https://rest.uniprot.org/uniprotkb/search?...&cursor=ABC123&size=500>; rel="next"
```

We write a tiny parser for that header, then a **generator** that yields one page at a
time — memory-friendly even for huge result sets.

In [8]:
def get_next_link(headers):
    """Return the 'next' page URL from the Link header, or None if this is the last page."""
    link = headers.get("Link")
    if link:
        match = re.match(r'<(.+)>; rel="next"', link)
        if match:
            return match.group(1)
    return None


def iterate_search(query, fields=None, fmt="json", page_size=500):
    """Yield one response object per page, following the cursor until exhausted."""
    params = {"query": query, "format": fmt, "size": page_size}
    if fields:
        params["fields"] = fields
    url = f"{API}/uniprotkb/search"
    while url:
        r = requests.get(url, params=params)
        r.raise_for_status()
        yield r
        url = get_next_link(r.headers)
        params = None   # the next-page URL already encodes query/format/fields/cursor

In [9]:
# Demo: page through hormone proteins, 200 at a time, stopping early for the tutorial.
total, seen = None, 0
for page in iterate_search("keyword:KW-0372 AND reviewed:true",
                           fields="accession,gene_names", page_size=200):
    if total is None:
        total = int(page.headers["x-total-results"])
        print("total matches:", total)
    seen += len(page.json()["results"])
    print(f"  fetched page -> {seen}/{total}")
    if seen >= 400:      # stop after two pages so the demo stays quick
        print("  (stopping early)")
        break

total matches: 1876
  fetched page -> 200/1876
  fetched page -> 400/1876
  (stopping early)


## 8. Making requests robust

For real pipelines, harden the calls:

* **Reuse a `requests.Session`** — keeps the TCP connection alive across requests (faster).
* **Send a `User-Agent`** identifying your tool/contact — good API citizenship, and it
  helps the maintainers reach you if something misbehaves.
* **Retry transient failures** (`429 Too Many Requests`, `5xx`) with exponential backoff,
  honouring a `Retry-After` header when present.
* **`raise_for_status()`** so genuine errors surface loudly instead of silently
  returning an error page.

In [10]:
session = requests.Session()
session.headers.update({
    "User-Agent": "uniprot-tutorial/1.0 (you@example.com)",  # <- put your contact here
})

RETRYABLE = {429, 500, 502, 503, 504}

def get(url, max_retries=5, **kwargs):
    """Session GET with backoff on transient errors and raise_for_status on real ones."""
    for attempt in range(max_retries):
        r = session.get(url, **kwargs)
        if r.status_code in RETRYABLE:
            wait = float(r.headers.get("Retry-After", 2 ** attempt))
            print(f"  {r.status_code} -> retry in {wait:.0f}s "
                  f"({attempt + 1}/{max_retries})")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    r.raise_for_status()   # last response was still retryable -> surface it
    return r


def fetch_all(query, fields=None, fmt="json", page_size=500):
    """Collect every result for a query using the robust get() + cursor pagination."""
    params = {"query": query, "format": fmt, "size": page_size}
    if fields:
        params["fields"] = fields
    url, records = f"{API}/uniprotkb/search", []
    while url:
        r = get(url, params=params)
        records.extend(r.json()["results"])
        url = get_next_link(r.headers)
        params = None
    return records


records = fetch_all("gene:INS AND reviewed:true", fields="accession,id,gene_names")
print("fetched", len(records), "records")

fetched 97 records


## 9. The `stream` endpoint — bulk downloads

For large exports, the **`/uniprotkb/stream`** endpoint returns the *entire* result set
in a single response — no paging. Trade-offs:

* **Use `stream`** when you want all results of a (large) query in one file.
* **Use cursor pagination (`/search`)** when you want to process results incrementally,
  show progress, or keep memory flat.

With `requests`, pass `stream=True` and consume `iter_content` so the whole payload
isn't buffered in memory at once.

In [11]:
params = {"query": "gene:INS AND reviewed:true", "format": "fasta"}

with session.get(f"{API}/uniprotkb/stream", params=params, stream=True) as r:
    r.raise_for_status()
    r.encoding = "utf-8"
    parts = [chunk for chunk in r.iter_content(chunk_size=8192, decode_unicode=True)]

fasta_text = "".join(parts)
print(fasta_text[:300], "...\n")
print("entries streamed:", fasta_text.count(">"))

>sp|O22765|TRPA1_ARATH Tryptophan synthase alpha chain OS=Arabidopsis thaliana OX=3702 GN=TRPA1 PE=1 SV=2
MDLLKTPSSTVGLSETFARLKSQGKVALIPYITAGDPDLSTTAKALKVLDSCGSDIIELG
VPYSDPLADGPAIQAAARRSLLKGTNFNSIISMLKEVIPQLSCPIALFTYYNPILRRGVE
NYMTVIKNAGVHGLLVPDVPLEETETLRNEARKHQIELVLLTTPTTPKERMNAIVEASEG
FIYLVSSVGVT ...

entries streamed: 97


## 10. The ID-mapping API (asynchronous)

UniProt's [ID mapping](https://www.uniprot.org/help/id_mapping) converts identifiers
between databases — e.g. UniProt accessions → PDB / RefSeq, or gene names → accessions.
Unlike search, it's **asynchronous**: you *submit* a job, *poll* until it's ready, then
*fetch* the results.

```
POST /idmapping/run            -> { "jobId": ... }
GET  /idmapping/status/{job}   -> { "jobStatus": "RUNNING" | "FINISHED" }
GET  /idmapping/results/{job}  -> { "results": [ {from, to}, ... ] }   (paginated!)
```

Database names (`from`/`to`) come from the
[id-mapping fields docs](https://www.uniprot.org/help/id_mapping) — e.g.
`UniProtKB_AC-ID`, `PDB`, `RefSeq_Protein`, `Gene_Name`.

In [12]:
def submit_id_mapping(from_db, to_db, ids):
    """Submit a mapping job; returns its jobId."""
    r = session.post(f"{API}/idmapping/run",
                     data={"from": from_db, "to": to_db, "ids": ",".join(ids)})
    r.raise_for_status()
    return r.json()["jobId"]


def wait_for_job(job_id, poll_every=2, timeout=60):
    """Poll status until the job is ready (with a simple timeout)."""
    url = f"{API}/idmapping/status/{job_id}"
    waited = 0
    while True:
        status = session.get(url).json()
        # UniProt->UniProt jobs return results inline; other targets report FINISHED.
        if "results" in status or "failedIds" in status:
            return
        if status.get("jobStatus") == "FINISHED":
            return
        if status.get("jobStatus") not in (None, "RUNNING", "NEW"):
            raise RuntimeError(f"job failed: {status}")
        if waited >= timeout:
            raise TimeoutError(f"job {job_id} not ready after {timeout}s")
        time.sleep(poll_every)
        waited += poll_every


def get_id_mapping_results(job_id, page_size=500):
    """Fetch all mapping pairs, following the Link header (results paginate too)."""
    url = f"{API}/idmapping/results/{job_id}"
    params, rows = {"format": "json", "size": page_size}, []
    while url:
        r = session.get(url, params=params)
        r.raise_for_status()
        rows.extend(r.json()["results"])
        url = get_next_link(r.headers)
        params = None
    return rows


job = submit_id_mapping("UniProtKB_AC-ID", "PDB", ["P01308", "P06213"])
print("jobId:", job)
wait_for_job(job)
pairs = get_id_mapping_results(job)
print(len(pairs), "PDB cross-references")
pd.DataFrame(pairs).head(8)

jobId: knQMRyYq6a
443 PDB cross-references


,from,to
0,P01308,1A7F
1,P01308,1AI0
2,P01308,1AIY
3,P01308,1B9E
4,P01308,1BEN
5,P01308,1EFE
6,P01308,1EV3
7,P01308,1EV6


## 11. Useful snippets

A few patterns that come up constantly.

In [13]:
# (1) Fetch ONE entry directly by accession (no search needed) in any format:
acc = "P01308"
print("direct fetch:\n" + session.get(f"{API}/uniprotkb/{acc}.fasta").text)

# (2) Count-only query (size=0 -> empty body, total in the header). Very cheap.
def count_hits(query):
    r = session.get(f"{API}/uniprotkb/search",
                    params={"query": query, "format": "list", "size": 0})
    r.raise_for_status()
    return int(r.headers["x-total-results"])

print("human reviewed proteins:", count_hits("organism_id:9606 AND reviewed:true"))

# (3) The same conventions (query/format/fields/pagination) work for OTHER datasets:
#     /uniref/search   /uniparc/search   /proteomes/search   /taxonomy/search
tax = session.get(f"{API}/taxonomy/search",
                  params={"query": "Homo sapiens", "format": "json", "size": 1}).json()
print("taxonomy lookup:", tax["results"][0]["scientificName"],
      "->", tax["results"][0]["taxonId"])

direct fetch:
>sp|P01308|INS_HUMAN Insulin OS=Homo sapiens OX=9606 GN=INS PE=1 SV=1
MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAED
LQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN

human reviewed proteins: 20431
taxonomy lookup: Homo sapiens -> 9606


## 12. Capstone — JSON → view → FASTA

Putting it together: query the **typical fields that make up a FASTA record** as JSON,
inspect them in a table, reconstruct UniProt-style FASTA headers, and save a `.fasta`
file — then verify it against UniProt's own FASTA output.

A UniProt FASTA header looks like:

```
>sp|P01308|INS_HUMAN Insulin OS=Homo sapiens OX=9606 GN=INS PE=1 SV=1
```

| token | meaning | JSON source |
|-------|---------|-------------|
| `sp`/`tr` | Swiss-Prot (reviewed) / TrEMBL | `entryType` |
| accession | `P01308` | `primaryAccession` |
| entry name | `INS_HUMAN` | `uniProtkbId` |
| protein name | `Insulin` | `proteinDescription.recommendedName.fullName.value` |
| `OS=` | organism | `organism.scientificName` |
| `OX=` | taxon id | `organism.taxonId` |
| `GN=` | gene name | `genes[0].geneName.value` |
| `PE=` | protein existence (1–5) | `proteinExistence` |
| `SV=` | sequence version | `entryAudit.sequenceVersion` |

### 12a. Query the FASTA fields as JSON

In [14]:
FASTA_FIELDS = ("accession,id,protein_name,organism_name,organism_id,"
                "gene_names,protein_existence,sequence_version,sequence")
CAPSTONE_QUERY = "gene:INS AND organism_id:9606 AND reviewed:true"

r = get(f"{API}/uniprotkb/search",
        params={"query": CAPSTONE_QUERY, "format": "json",
                "fields": FASTA_FIELDS, "size": 5})
records = r.json()["results"]
print(len(records), "records")
records[0]   # one raw, nested JSON record

2 records


{'entryType': 'UniProtKB reviewed (Swiss-Prot)',
 'primaryAccession': 'P01308',
 'uniProtkbId': 'INS_HUMAN',
 'entryAudit': {'sequenceVersion': 1},
 'organism': {'scientificName': 'Homo sapiens',
  'commonName': 'Human',
  'taxonId': 9606,
  'lineage': ['Eukaryota',
   'Metazoa',
   'Chordata',
   'Craniata',
   'Vertebrata',
   'Euteleostomi',
   'Mammalia',
   'Eutheria',
   'Euarchontoglires',
   'Primates',
   'Haplorrhini',
   'Catarrhini',
   'Hominidae',
   'Homo']},
 'proteinExistence': '1: Evidence at protein level',
 'proteinDescription': {'recommendedName': {'fullName': {'value': 'Insulin'}},
  'contains': [{'recommendedName': {'fullName': {'value': 'Insulin B chain'}}},
   {'recommendedName': {'fullName': {'value': 'Insulin A chain'}}}],
  'flag': 'Precursor'},
 'genes': [{'geneName': {'value': 'INS'}}],
 'sequence': {'value': 'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN',
  'length': 110,
  'molWeight': 119

### 12b. Flatten the nested JSON and view it

In [15]:
def parse_record(rec):
    """Pull the flat FASTA-header fields out of one nested JSON record."""
    desc = rec.get("proteinDescription", {})
    name = desc.get("recommendedName", {}).get("fullName", {}).get("value", "")
    genes = rec.get("genes", [])
    gene = genes[0]["geneName"]["value"] if genes and "geneName" in genes[0] else ""
    db = "sp" if "Swiss-Prot" in rec.get("entryType", "") else "tr"
    pe = rec.get("proteinExistence", "").split(":")[0].strip()  # "1: Evidence..." -> "1"
    return {
        "db": db,
        "accession": rec["primaryAccession"],
        "entry_name": rec["uniProtkbId"],
        "protein_name": name,
        "organism": rec["organism"]["scientificName"],
        "ox": rec["organism"]["taxonId"],
        "gene": gene,
        "pe": pe,
        "sv": rec.get("entryAudit", {}).get("sequenceVersion", ""),
        "sequence": rec["sequence"]["value"],
    }

parsed = [parse_record(rec) for rec in records]
pd.DataFrame(parsed)[["db", "accession", "entry_name", "protein_name",
                      "organism", "ox", "gene", "pe", "sv"]]

,db,accession,entry_name,protein_name,organism,ox,gene,pe,sv
0,sp,P01308,INS_HUMAN,Insulin,Homo sapiens,9606,INS,1,1
1,sp,F8WCM5,INSR2_HUMAN,"Insulin, isoform 2",Homo sapiens,9606,INS-IGF2,1,1


### 12c. Reconstruct FASTA records and save to a file

In [16]:
def wrap_sequence(seq, width=60):
    """Wrap a sequence at `width` chars per line, as in standard FASTA."""
    return "\n".join(seq[i:i + width] for i in range(0, len(seq), width))


def record_to_fasta(rec):
    """Build a UniProtKB-style FASTA record from a parsed dict."""
    header = (f">{rec['db']}|{rec['accession']}|{rec['entry_name']} "
              f"{rec['protein_name']} "
              f"OS={rec['organism']} OX={rec['ox']}")
    if rec["gene"]:
        header += f" GN={rec['gene']}"
    header += f" PE={rec['pe']} SV={rec['sv']}"
    return header + "\n" + wrap_sequence(rec["sequence"])


def write_fasta(records, path):
    with open(path, "w") as fh:
        fh.write("\n".join(record_to_fasta(r) for r in records) + "\n")
    return path


print(record_to_fasta(parsed[0]))      # preview the first record
out_path = write_fasta(parsed, "insulin_human.fasta")
print("\nwrote ->", out_path)

>sp|P01308|INS_HUMAN Insulin OS=Homo sapiens OX=9606 GN=INS PE=1 SV=1
MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAED
LQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN

wrote -> insulin_human.fasta


### 12d. Verify against UniProt's own FASTA

In [17]:
# Ask UniProt for the same query in FASTA format and compare the first header.
direct = get(f"{API}/uniprotkb/search",
             params={"query": CAPSTONE_QUERY, "format": "fasta", "size": 5}).text

print("UniProt's header :", direct.splitlines()[0])
print("our header       :", record_to_fasta(parsed[0]).splitlines()[0])

with open(out_path) as fh:
    saved = fh.read()
print("\nrecords in file  :", saved.count(">"))

UniProt's header : >sp|P01308|INS_HUMAN Insulin OS=Homo sapiens OX=9606 GN=INS PE=1 SV=1
our header       : >sp|P01308|INS_HUMAN Insulin OS=Homo sapiens OX=9606 GN=INS PE=1 SV=1

records in file  : 2


## 13. Wrap-up & other things worth knowing

You now have the core toolkit: build a query, choose a format, select fields, paginate
robustly, stream bulk data, map IDs, and turn JSON into FASTA.

**Other important things to be aware of:**

* **Be a good API citizen.** Send a `User-Agent` with a contact, don't hammer the server
  in tight loops, and prefer one large paginated/streamed request over thousands of tiny
  ones. Respect `Retry-After` on `429`.
* **Errors are informative.** On a bad query UniProt returns a 4xx with a JSON body
  explaining what went wrong — print `r.text` when `raise_for_status()` fires.
* **`search` vs `stream`.** `search` = incremental + progress + flat memory;
  `stream` = everything at once, simplest for bulk files.
* **ID mapping is asynchronous** — always submit → poll → fetch, and remember its results
  paginate too.
* **Prefer stable identifiers** in queries: `organism_id:9606` beats `organism:human`;
  `accession:` and `proteome:` are unambiguous.
* **Caching.** Responses carry `Cache-Control`; for repeated runs consider
  [`requests-cache`](https://requests-cache.readthedocs.io/) to avoid refetching.
* **Other datasets** share these exact conventions: `uniref`, `uniparc`, `proteomes`,
  `taxonomy`, `keywords`, `locations`.

**Reference docs**

* API queries — <https://www.uniprot.org/help/api_queries>
* Query fields — <https://www.uniprot.org/help/query-fields>
* Return fields — <https://www.uniprot.org/help/return_fields>
* Pagination — <https://www.uniprot.org/help/pagination>
* ID mapping — <https://www.uniprot.org/help/id_mapping>


## Including isoforms (`includeIsoform`)

By default a UniProtKB search returns only the **canonical** sequence of each entry.
Set `includeIsoform=true` to also get the **isoforms** (alternative splice products),
which come back as separate hits with `-N` suffixed accessions, e.g. `P05067-2`.

A clear example is **P05067** (APP, amyloid-beta precursor protein), which has many
isoforms. Notes:

* This parameter applies to **UniProtKB searches only**.
* `requests` serializes Python `True`/`False` to the strings `"True"`/`"False"`
  (which UniProt happens to accept case-insensitively), but pass the lowercase
  strings `"true"`/`"false"` to match the documented form.

In [18]:
def search_accessions(query, include_isoform):
    """Return the list of accessions for a query, with isoforms on or off."""
    r = session.get(f"{API}/uniprotkb/search",
                    params={"query": query, "format": "list",
                            "includeIsoform": include_isoform})
    r.raise_for_status()
    return r.text.split()

query = "accession:P05067"   # APP — amyloid-beta precursor protein
canonical = search_accessions(query, "false")
with_iso = search_accessions(query, "true")

print("includeIsoform=false :", len(canonical), "hit ->", canonical)
print("includeIsoform=true  :", len(with_iso), "hits ->", with_iso)
print("\nextra isoforms        :", sorted(set(with_iso) - set(canonical)))

includeIsoform=false : 1 hit -> ['P05067']
includeIsoform=true  : 11 hits -> ['P05067-4', 'P05067', 'P05067-2', 'P05067-8', 'P05067-10', 'P05067-5', 'P05067-7', 'P05067-11', 'P05067-6', 'P05067-3', 'P05067-9']

extra isoforms        : ['P05067-10', 'P05067-11', 'P05067-2', 'P05067-3', 'P05067-4', 'P05067-5', 'P05067-6', 'P05067-7', 'P05067-8', 'P05067-9']


### Retrieving an isoform's changed sequence

Listing isoform accessions is only half the story — you usually want the isoform's
**actual (changed) amino-acid sequence**. Fetch it by querying the isoform accession
directly (`P05067-2`), in any format. Below we grab the canonical and isoform sequences
and confirm they differ in length.

*What* changed is annotated on the **canonical** entry as `VAR_SEQ` ("Alternative
sequence") features (`fields=ft_var_seq`): each gives the residue range, which isoform
it applies to, and the original → alternative residues (or a deletion). That's UniProt's
own, biologically meaningful description of the splice change — far better than naively
diffing two strings.

In [19]:
def get_sequence(accession):
    """Fetch the FASTA header and amino-acid sequence for any accession (incl. isoforms)."""
    lines = session.get(f"{API}/uniprotkb/{accession}.fasta").text.splitlines()
    return lines[0], "".join(lines[1:])


def report_isoform(canonical, isoform):
    """Print the isoform's changed sequence next to the canonical, and return the
    canonical entry's VAR_SEQ ("Alternative sequence") features as a DataFrame.

    1) The isoform's OWN (changed) sequence is fetched by its -N accession.
    2) *What* changed is annotated on the CANONICAL entry as VAR_SEQ features
       (fields=ft_var_seq): residue range, which isoform it applies to, and the
       original -> alternative residues (or a deletion) -- UniProt's own,
       biologically meaningful description of the splice change.
    """
    iso_header, iso_seq = get_sequence(isoform)
    _, canon_seq = get_sequence(canonical)
    print(iso_header)
    print(f"canonical {canonical} : {len(canon_seq)} aa")
    print(f"isoform {isoform} : {len(iso_seq)} aa")
    print(f"isoform sequence (first 80 aa): {iso_seq[:80]}")

    feats = session.get(f"{API}/uniprotkb/{canonical}.json",
                        params={"fields": "ft_var_seq"}).json().get("features", [])
    rows = []
    for f in feats:
        if f.get("type") != "Alternative sequence":
            continue
        alt = f.get("alternativeSequence", {})
        rows.append({
            "feature": f.get("featureId"),
            "from": f["location"]["start"]["value"],
            "to": f["location"]["end"]["value"],
            "isoform": f.get("description", ""),
            "original": alt.get("originalSequence") or "(none)",
            "becomes": (alt.get("alternativeSequences") or ["(deleted)"])[0],
        })

    df = pd.DataFrame(rows)
    print(f"\n{len(rows)} VAR_SEQ (alternative sequence) features:")
    return df


# Example 1: APP (amyloid-beta precursor protein), isoform APP305
report_isoform("P05067", "P05067-2").head(8)

>sp|P05067-2|A4_HUMAN Isoform APP305 of Amyloid-beta precursor protein OS=Homo sapiens OX=9606 GN=APP
canonical P05067 : 770 aa
isoform P05067-2 : 305 aa
isoform sequence (first 80 aa): MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMNVQNGKWDSDPSGTKTCIDTKEGILQYCQEVYPEL

13 VAR_SEQ (alternative sequence) features:


,feature,from,to,isoform,original,becomes
0,VSP_045446,1,19,in isoform 11,MLPGLALLLLAAWTARALE,MDQLEDLLVLFINY
1,VSP_009116,19,74,in isoform APP639,(none),(deleted)
2,VSP_009117,289,363,in isoform APP639,(none),(deleted)
3,VSP_000002,289,289,"in isoform APP695, isoform L-APP696, isoform L...",E,V
4,VSP_000004,290,364,in isoform APP695 and isoform L-APP677,(none),(deleted)
5,VSP_000003,290,345,in isoform L-APP696 and isoform APP714,(none),(deleted)
6,VSP_000005,290,305,in isoform APP305,VCSEQAETGPCRAMIS,KWYKEVHSGQARWLML
7,VSP_000006,306,770,in isoform APP305,(none),(deleted)


Because the logic now lives in `report_isoform`, applying it to any other entry is a
one-liner. Here's a second example: **Q9JKS4** (LDB3, LIM domain-binding protein 3, in
mouse), which also has several splice isoforms — we look at `Q9JKS4-2`.

In [20]:
# Example 2: LDB3 / LIM domain-binding protein 3 (mouse)
report_isoform("Q9JKS4", "Q9JKS4-2").head(8)

>sp|Q9JKS4-2|LDB3_MOUSE Isoform 2 of LIM domain-binding protein 3 OS=Mus musculus OX=10090 GN=Ldb3
canonical Q9JKS4 : 723 aa
isoform Q9JKS4-2 : 684 aa
isoform sequence (first 80 aa): MSYSVTLTGPGPWGFRLQGGKDFNMPLTISRITPGSKAAQSQLSQGDLVVAIDGVNTDTMTHLEAQNKIKSASYNLSLTL

4 VAR_SEQ (alternative sequence) features:


,feature,from,to,isoform,original,becomes
0,VSP_051903,108,227,"in isoform 2, isoform 4 and isoform 6",DPALDTNGSLATPSPSPEARASPGALEFGDTFSSSFSQTSVCSPLM...,VVANSPANADYQERFNPSVLKDSALSTHKPIEVKGLGGKATIIHAQ...
1,VSP_051904,296,357,in isoform 3 and isoform 4,(none),(deleted)
2,VSP_051905,296,327,in isoform 5 and isoform 6,STPIEHAPVCTSQATSPLLPASAQSPAAASPI,RERFETERNSPRFAKLRNWHHGLSAQILNVKS
3,VSP_051906,328,723,in isoform 5 and isoform 6,(none),(deleted)
